# Cycle 2 — Hyperparameter Tuning: xG Model

**Project:** Football Predictor  
**Depends on:** `cycle2_modelling.ipynb`

---

## Baseline Results (from cycle2_modelling.ipynb)

| Model | Accuracy | AUC-ROC |
|-------|----------|----------|
| Dummy | 89.18% | 0.5000 |
| Logistic Regression | 72.50% | 0.7963 |
| Random Forest | 88.82% | 0.7884 |
| XGBoost | 82.32% | 0.7871 |
|LightGBM|77.47%|0.8035|

## What We Are Tuning and Why

We tune XGBoost and Random Forest and LightGBM.

**Scoring metric: AUC-ROC** — not accuracy. The tuner will search for the combination that maximises AUC, which is the correct objective for an imbalanced binary classification problem.

In [6]:
import sys, os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
import joblib, os

# Locate project root (folder containing data/, models/, notebooks/)
_here = os.getcwd()
while not os.path.isdir(os.path.join(_here, 'data')):
    _p = os.path.dirname(_here)
    if _p == _here: raise RuntimeError('project root not found')
    _here = _p
if _here not in sys.path:
    sys.path.insert(0, _here)

from config import Paths, ensure_dirs
ensure_dirs()  # creates models/cycle1-3 if missing


In [7]:
df = pd.read_csv(str(Paths.WYSCOUT_PROCESSED))
X = df.drop(columns=['Goal'])
y = df['Goal']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

neg, pos = y_train.value_counts()[0], y_train.value_counts()[1]
scale_pos_weight = neg / pos

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'scale_pos_weight: {scale_pos_weight:.2f}')
print(f'CV: StratifiedKFold(n_splits=5)')

Train: 6,760 | Test: 1,691
scale_pos_weight: 8.25
CV: StratifiedKFold(n_splits=5)


## Tune XGBoost

In [8]:
xgb_param_grid = {
    'n_estimators':     [100, 200, 300, 500],
    'max_depth':        [3, 4, 5, 6],
    'learning_rate':    [0.01, 0.05, 0.1, 0.2],
    'subsample':        [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma':            [0, 0.1, 0.2],
    'scale_pos_weight': [scale_pos_weight, scale_pos_weight * 0.5, scale_pos_weight * 1.5],
}

total = 4*4*4*3*3*3*3*3
print(f'Total possible combinations: {total:,}')
print(f'We will try: 50 (RandomizedSearch x 5-fold CV = 250 fits)')

Total possible combinations: 15,552
We will try: 50 (RandomizedSearch x 5-fold CV = 250 fits)


In [9]:
xgb = XGBClassifier(random_state=42, eval_metric='auc', verbosity=0)

search_xgb = RandomizedSearchCV(
    xgb, xgb_param_grid,
    n_iter=50,
    cv=cv,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
    verbose=1
)
search_xgb.fit(X_train_s, y_train)

print('Best hyperparameters:')
for k, v in search_xgb.best_params_.items():
    print(f'  {k}: {v}')
print()
print(f'Best CV AUC:   {search_xgb.best_score_:.4f}')

y_prob_xgb_tuned = search_xgb.best_estimator_.predict_proba(X_test_s)[:, 1]
y_pred_xgb_tuned = search_xgb.best_estimator_.predict(X_test_s)
print(f'Test AUC:      {roc_auc_score(y_test, y_prob_xgb_tuned):.4f}')
print(f'Test Accuracy: {accuracy_score(y_test, y_pred_xgb_tuned)*100:.2f}%')

Fitting 5 folds for each of 50 candidates, totalling 250 fits


Best hyperparameters:
  subsample: 0.7
  scale_pos_weight: 8.247606019151847
  n_estimators: 200
  min_child_weight: 5
  max_depth: 4
  learning_rate: 0.01
  gamma: 0
  colsample_bytree: 0.7

Best CV AUC:   0.8137
Test AUC:      0.8183
Test Accuracy: 73.39%


### Observations:
- **Test AUC 0.8183** — a jump of +0.0312 from untuned XGBoost (0.7871)
- Low learning rate (0.01) + 200 trees again — same pattern as Cycle 1
- scale_pos_weight stayed at the computed 8.25 — the default imbalance ratio was already optimal
- min_child_weight=5 (conservative) — prevents overfitting on the minority Goal class
- Test AUC (0.8183) slightly above CV AUC (0.8137) — model generalises well

### Improvement over baseline:
- Untuned XGBoost:  AUC 0.7871
- Tuned XGBoost:    AUC 0.8183
- **Gain: +0.0312**

In [10]:
print('TUNED XGBOOST — Full Report')
print(f'AUC-ROC:  {roc_auc_score(y_test, y_prob_xgb_tuned):.4f}')
print(f'Accuracy: {accuracy_score(y_test, y_pred_xgb_tuned)*100:.2f}%')
print()
print(classification_report(y_test, y_pred_xgb_tuned, target_names=['No Goal', 'Goal']))

TUNED XGBOOST — Full Report
AUC-ROC:  0.8183
Accuracy: 73.39%

              precision    recall  f1-score   support

     No Goal       0.96      0.73      0.83      1508
        Goal       0.26      0.76      0.38       183

    accuracy                           0.73      1691
   macro avg       0.61      0.75      0.61      1691
weighted avg       0.89      0.73      0.78      1691



### Observations:
- Goal recall 0.76 — model correctly identifies 76% of actual goals
- Goal precision 0.26 — of all shots predicted as Goal, 26% actually are goals (expected given 10.8% base rate)
- The precision/recall trade-off is appropriate for an xG model — we want high recall (catch most goals) at the cost of some false positives
- AUC 0.8183 is a strong result for xG

## Tune Random Forest

In [16]:
rf_param_grid = {
    'n_estimators':      [100, 200, 300, 500],
    'max_depth':         [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2', None],
    'class_weight':      ['balanced', 'balanced_subsample'],
}

rf = RandomForestClassifier(random_state=42)

search_rf = RandomizedSearchCV(
    rf, rf_param_grid,
    n_iter=50,
    cv=cv,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
    verbose=1
)
search_rf.fit(X_train_s, y_train)

print('Best hyperparameters:')
for k, v in search_rf.best_params_.items():
    print(f'  {k}: {v}')
print()
print(f'Best CV AUC:   {search_rf.best_score_:.4f}')

y_prob_rf_tuned = search_rf.best_estimator_.predict_proba(X_test_s)[:, 1]
y_pred_rf_tuned = search_rf.best_estimator_.predict(X_test_s)
print(f'Test AUC:      {roc_auc_score(y_test, y_prob_rf_tuned):.4f}')
print(f'Test Accuracy: {accuracy_score(y_test, y_pred_rf_tuned)*100:.2f}%')
print()
print(classification_report(y_test, y_pred_rf_tuned, target_names=['No Goal', 'Goal']))

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best hyperparameters:
  n_estimators: 300
  min_samples_split: 10
  min_samples_leaf: 2
  max_features: log2
  max_depth: 5
  class_weight: balanced_subsample

Best CV AUC:   0.8120
Test AUC:      0.8176
Test Accuracy: 74.16%

              precision    recall  f1-score   support

     No Goal       0.96      0.74      0.84      1508
        Goal       0.26      0.77      0.39       183

    accuracy                           0.74      1691
   macro avg       0.61      0.75      0.61      1691
weighted avg       0.89      0.74      0.79      1691



### Observations

- Tuned RF AUC 0.8176 — very close to tuned XGBoost (0.8183), only 0.0007 behind
- balanced_subsample (different class weight per tree) chosen over balanced — consistent with Cycle 1 RF tuning result\n- max_depth=5 (shallow) — prevents overfitting, RF with deep trees tends to overfit on small minority classes
- Goal recall 0.77 — marginally higher than XGBoost, but lower precision (0.26)

### Improvement over baseline
- Untuned RF:  AUC 0.7884
- Tuned RF:    AUC 0.8176
- **Gain: +0.0292**

## Tune LightGBM

In [12]:
lgb_param_grid = {
    'n_estimators':     [100, 200, 300, 500],
    'learning_rate':    [0.01, 0.05, 0.1, 0.15],
    'max_depth':        [3, 4, 5, 6, 7],
    'num_leaves':       [20, 31, 50, 100],
    'subsample':        [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_samples':[10, 20, 30],
    'scale_pos_weight': [scale_pos_weight, scale_pos_weight * 0.5, scale_pos_weight * 1.5],
}

total_lgb = 4 * 4 * 5 * 4 * 4 * 4 * 3 * 3
print(f'Total possible LightGBM combinations: {total_lgb:,}')
print(f'We will try: 50 (RandomizedSearch x 5-fold CV = 250 fits)')

Total possible LightGBM combinations: 46,080
We will try: 50 (RandomizedSearch x 5-fold CV = 250 fits)


In [13]:
lgb = LGBMClassifier(random_state=42, verbose=-1)

search_lgb = RandomizedSearchCV(
    lgb, lgb_param_grid,
    n_iter=50,
    cv=cv,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
    verbose=1
)
search_lgb.fit(X_train_s, y_train)

print('Best hyperparameters:')
for k, v in search_lgb.best_params_.items():
    print(f'  {k}: {v}')
print()
print(f'Best CV AUC:   {search_lgb.best_score_:.4f}')

y_prob_lgb_tuned = search_lgb.best_estimator_.predict_proba(X_test_s)[:, 1]
y_pred_lgb_tuned = search_lgb.best_estimator_.predict(X_test_s)
print(f'Test AUC:      {roc_auc_score(y_test, y_prob_lgb_tuned):.4f}')
print(f'Test Accuracy: {accuracy_score(y_test, y_pred_lgb_tuned)*100:.2f}%')

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best hyperparameters:
  subsample: 0.8
  scale_pos_weight: 12.371409028727772
  num_leaves: 100
  n_estimators: 500
  min_child_samples: 10
  max_depth: 3
  learning_rate: 0.01
  colsample_bytree: 0.7

Best CV AUC:   0.8127
Test AUC:      0.8152
Test Accuracy: 64.93%


### Improvement over baseline
- Untuned LGBM:  AUC 0.8035
- Tuned RF:    AUC 0.8152

In [14]:
print('TUNED LIGHTGBM — Full Report')
print(f'AUC-ROC:  {roc_auc_score(y_test, y_prob_lgb_tuned):.4f}')
print(f'Accuracy: {accuracy_score(y_test, y_pred_lgb_tuned)*100:.2f}%')
print()
print(classification_report(y_test, y_pred_lgb_tuned, target_names=['No Goal', 'Goal']))

TUNED LIGHTGBM — Full Report
AUC-ROC:  0.8152
Accuracy: 64.93%

              precision    recall  f1-score   support

     No Goal       0.98      0.62      0.76      1508
        Goal       0.22      0.87      0.35       183

    accuracy                           0.65      1691
   macro avg       0.60      0.75      0.55      1691
weighted avg       0.89      0.65      0.72      1691



# Full Results Comparison

### Key Conclusions

**1. Tuning makes a major difference**
- XGBoost: 0.7871 → 0.8183 (+0.0312)
- Random Forest: 0.7884 → 0.8176 (+0.0292)
- LightGBM: 0.8035 → 0.8152 (+0.0117)

**2. Tuned models beat Logistic Regression**
Both tuned models exceed the LR baseline (0.7963), which led untuned. This confirms that while LR is a strong default for xG, gradient boosting surpasses it with proper tuning.

**3. Accuracy is not the metric**

All tuned models show accuracy around 73-74% — lower than the 89% dummy. This is correct behaviour: the models are trading some false negatives (predicting No Goal when it is Goal) for high recall on actual goals.

# Save Best Model

In [17]:
# Determine best model by AUC
xgb_auc = roc_auc_score(y_test, y_prob_xgb_tuned)
rf_auc  = roc_auc_score(y_test, y_prob_rf_tuned)
lgb_auc     = roc_auc_score(y_test, y_prob_lgb_tuned)

best_auc   = max(xgb_auc, rf_auc, lgb_auc)
best_model = {xgb_auc: search_xgb.best_estimator_, rf_auc: search_rf.best_estimator_, lgb_auc: search_lgb.best_estimator_}[best_auc]
best_name  = {xgb_auc: 'XGBoost Tuned', rf_auc: 'Random Forest Tuned', lgb_auc: 'LightGBM'}[best_auc]

print(f'Saving: {best_name} (AUC={best_auc:.4f})')


joblib.dump(best_model,           str(Paths.C2_MODEL))
joblib.dump(scaler,               str(Paths.C2_SCALER))
joblib.dump(list(X_train.columns),str(Paths.C2_FEATURES))

print('Saved: ../../models/cycle2_best_model.pkl')
print('Saved: ../../models/cycle2_scaler.pkl')
print('Saved: ../../models/cycle2_feature_cols.pkl')

Saving: XGBoost Tuned (AUC=0.8183)
Saved: ../../models/cycle2_best_model.pkl
Saved: ../../models/cycle2_scaler.pkl
Saved: ../../models/cycle2_feature_cols.pkl


### Notes for Report

- Three artefacts saved: model, scaler, feature column list
- Same modular pattern as Cycle 1 — each component can be updated independently
- Features: ['X', 'Y', 'Distance', 'Angle', 'Left_Foot', 'Right_Foot', 'Header', 'First_Half', 'Player_Rank']
- The API will load these three files at startup to serve xG predictions